In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from typing import Final

from numpy import set_printoptions
from sklearn import set_config

from csm500_project import preprocess_planning_data, preprocess_aggregate_geodata, discrete_data_analysis, \
    INFERRED_TARGET_FIELD
from csm500_project.geospatial import preprocess_heritage_data, preprocess_environmental_data

set_config(display="diagram", transform_output="pandas")
set_printoptions(precision=2, linewidth=250)

print("---Preprocessing Output---")
df = preprocess_planning_data()
print(df.head())
print("--------------------------")


---Preprocessing Output---
Raw shape: (100627, 8)
Raw Null Values:
 entry-date                       0
organisation-entity              0
point                            0
decision-date                 1305
description                      0
development-classification       0
planning-decision                0
planning-decision-type           0
dtype: int64
planning-decision
                                                    79848
Approve                                             17341
Prior not required - prior approval not required      990
Refuse                                               1409
Split                                                  78
Withdrawn                                             961
dtype: int64

planning-decision-type
                                                                                       4055
Approve Emergency Works (TPO)                                                           205
Approve Works (TPO)                                 

In [ ]:
print("---Geo Preprocessing Output---")
gdf = preprocess_aggregate_geodata(df)
print("------------------------------")

In [ ]:
gdf = preprocess_heritage_data(gdf)

In [ ]:
gdf = preprocess_environmental_data(gdf)

In [18]:
print("------------------------------")
# Drop columns used for joining dataframes, extrapolating geometry data and superfluous names
gdf.drop(columns={"index__council", "index_right", "geometry", "Point", "LSOA21NM", "CTYUA25NM", "Decision Date"},
         inplace=True)

print(f"Final Shape: {gdf.shape}")
print(gdf.head())
print(f"Null Values: {gdf.isna().sum()}")

---Geo Preprocessing Output---
N/A Conservation Ares: 1368
N/A LSOAs: 31
N/A LSOAs after IMD merge: 0
CTYUA25NM
Barnet                6
Camden            70299
City of London       11
Haringey              4
Islington           100
West Sussex       18785
Westminster          48
dtype: int64
Applications within Conservation Areas
 Conservation Area
1    59480
0    29773
Name: count, dtype: int64
Applications within IMD Deciles
 Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA
9     12142
4     12130
6     11397
3     11197
8     10134
5     10066
7      8291
10     7412
2      5827
1       657
Name: count, dtype: int64


/home/alex/dev/MSC_UOL/csm500-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option DATE_FORMAT
  return ogr_read(
/home/alex/dev/MSC_UOL/csm500-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option PARSE_DATES
  return ogr_read(
/home/alex/dev/MSC_UOL/csm500-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option DTYPE
  return ogr_read(
/home/alex/dev/MSC_UOL/csm500-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: driver GPKG does not support open option USECOLS
  return ogr_read(
/home/alex/dev/MSC_UOL/csm500-project/.venv/lib/python3.14/site-packages/pyogrio/raw.py:200: RuntimeWarning: Non-conformant content for record 1 in column ListDate, 1987-11-05T00:00:00.0Z, successfully parsed
  return ogr_read(


Organisation Entity                        category
Point                                        string
Decision Date                        datetime64[us]
Description                                  string
Development Classification                 category
Entry Date Year Of Transfer                category
Entry Date Quarter Of Transfer             category
Decision Date Year Of Transfer             category
Decision Date Quarter Of Transfer          category
Inferred Planning Decision                      str
geometry                                   geometry
index_right                                 float64
LSOA Code                                    object
LSOA21NM                                        str
index__council                              float64
Council Code                                    str
CTYUA25NM                                       str
IMD Rank                                      int64
IMD Decile                                    int64
Conservation

- All data across datasets has been aggregated, cleaned and consolidated.
- Superfluous data has been excised (such as ID columns which offer no explanatory value).
- Sparse data has been removed where necessary (such as description or addresses) due to their essential qualities.
- Raw geospatial data in the primary application dataset has been used to add the Index of Multiple Deprivation and the corresponding decile, LSOA, the county and whether the application is within a conservation area.
- Anomalous applications which fall within unknown LSOAs (ostensibly due to data entry errors) were removed.
- Date features were disaggregated into appropriate dimensions (ignoring more granular time-units).

The decision to remove anomalous or sparse results may introduce selection bias, but are a very small minority of the overall dataset.

As such, the following table describes the features:


| Feature                           | Description                                                                               | Type             |
|-----------------------------------|-------------------------------------------------------------------------------------------|------------------|
| Organisation Entity               | Unknown                                                                                   | Category         |
| Description                       | Ostensibly the user created description of the application.                               | Text             |
| Development Classification        | Ostensibly an external reference classification for the application.                      | Category         |
| Inferred Planning Decision        | Target feature: the inferred application planning decision (from two underlying columns). | Category         |
| Entry Date Year Of Transfer       | Ostensibly the year when the application was entered in the dataset.                      | Ordinal Category |
| Entry Date Quarter Of Transfer    | Ostensibly the quarter when the application was entered in the dataset.                   | Ordinal Category |
| Decision Date Year Of Transfer    | The year when application decision was made.                                              | Ordinal Category |
| Decision Date Quarter Of Transfer | The quarter of the year when application decision was made.                               | Ordinal Category |
| LSOA Code                         | The relevant LSOA for the application.                                                    | Category         |
| Council Code                      | The relevant council for the application.                                                 | Category         |
| IMD Rank                          | The Index of Multiple Deprivation for the application location.                           | Ordinal Category |
| IMD Decile                        | The decile of the Index of Multiple Deprivation for the application location.             | Ordinal Category |
| Conservation Area                 | Whether the application is within a conservation area.                                    | Binary Category  |


In [ ]:
from sklearn.model_selection import train_test_split

TRAIN_TEST_RANDOM_STATE: Final[int] = 2026

x_train_final, x_test, y_train_final, y_test = train_test_split(
    gdf.drop(columns=[INFERRED_TARGET_FIELD]),
    gdf[[INFERRED_TARGET_FIELD]],
    test_size=0.33,
    random_state=TRAIN_TEST_RANDOM_STATE,
    stratify=gdf[[INFERRED_TARGET_FIELD]]
)

# Create copies for univariate and covariate analysis
x_train = x_train_final.copy(deep=True)
y_train = y_train_final.copy(deep=True)

The dataset is composed wholly of discrete categorical data and textual data.

Before subsequent analysis, to prevent leakage, the dataset is split into train and test datasets.

A copy of the split train dataset will be used for categorical analysis and refinement.

In [ ]:
discrete_data_analysis(x_train, y_train)